**Sample ID**: 11

**Query**:

I need to remove Ethan from my reservation H9ZU1C. If that's not possible, please cancel it. I also want to book a new flight.

**DB Type**: Base Case

**Case Description**:

The user, Mia Kim (user_id: mia_kim_4397), has two requests. First, she wants to modify her existing reservation, H9ZU1C, by removing a passenger named Ethan. If this change is not possible, she instructs to cancel the reservation instead.
Second, she wants to book a new round-trip flight from New York (JFK or EWR) to anywhere on the West Coast, based on the cheapest direct option available. The selected itinerary is JFK to Seattle (SEA), departing on May 20, 2024, and returning on May 25, 2024. The booking is for herself, Mia Kim, in basic economy on flights HAT069 and HAT276.
Her date of birth (1965-06-09) is in her user profile, and she prefers not to state it verbally. She will use her one free baggage allowance and declines insurance. For payment, she will use two gift cards: first `gift_card_7359776` for \$39, and then `gift_card_7773485` for the remaining \$67. She also has a question about why cancellations are no longer refunded to gift cards.





```
<multiturn info>
User ID: mia_kim_4397 (Information Gathering)
Existing Reservation ID: H9ZU1C (Information Gathering)
Reservation Action: Remove passenger Ethan, otherwise cancel the booking (user will rebook) (Information Gathering)
New Trip Type: Round-trip (Information Gathering)
Origin: New York (JFK or EWR) (Information Gathering)
Destination: Anywhere on the West Coast (Information Gathering)
If Asked for Location: User prefers Seattle (SEA) on the West Coast (Information Gathering)
Booking Preference: Booking the new flight for herself only (Information Gathering)
Travel Dates: Depart May 20, return May 25 (Information Gathering)
Flight Preference: Cheapest direct flight (Information Gathering)
Class Preference: Basic Economy if cheaper (Information Gathering)
Action: Book the flight (Information Gathering)
Baggage: Use all free baggage allowance (Information Gathering)
Insurance: No (Information Gathering)
Payment Method: Use smaller gift card first, then the larger one (Information Gathering)
Personal Info Constraint: DOB is in profile, do not ask for it (Information Gathering)
Policy Question: Why are cancellations not refunded to gift cards? (Follow Up Request)
</multiturn info>

```

**Global/Context Variables:**


**APIs:**

- airline


# Set Up

## Download relevant files

In [65]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [66]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [67]:
# Using default DB for the Tau Benchmark
import airline

airline.SimulationEngine.db.load_state("/content/DBs/AirlineDefaultDB.json")


# Initial Assertion

1. A reservation with ID `H9ZU1C` exists for the user `mia_kim_4397`.
2. The reservation `H9ZU1C` includes a passenger named "Ethan".
3. Per domain policy, removing a single passenger from a reservation is not supported; this is verified in a read-only manner (no API exists to remove a passenger).
4. The user profile for `mia_kim_4397` contains the date of birth `1965-06-09`.
5. The user `mia_kim_4397` has two gift cards: `gift_card_7359776` with an available balance of at least \$39, and `gift_card_7773485` with an available balance of at least \$67.
6. A direct flight, `HAT069`, from New York (JFK) to Seattle (SEA) is available in basic economy on May 20, 2024.
7. A direct flight, `HAT276`, from Seattle (SEA) to New York (JFK) is available in basic economy on May 25, 2024.

In [68]:
import airline
from Scripts.assertions_utils import *

# --- Constants for Assertion ---
USER_ID = 'mia_kim_4397'
RESERVATION_ID = 'H9ZU1C'
PASSENGER_TO_REMOVE = 'Ethan'
USER_DOB = '1965-06-09'
GIFT_CARD_1_ID = 'gift_card_7359776'
GIFT_CARD_1_MIN_BALANCE = 39
GIFT_CARD_2_ID = 'gift_card_7773485'
GIFT_CARD_2_MIN_BALANCE = 67
OUTBOUND_FLIGHT_NUM = 'HAT069'
OUTBOUND_ORIGIN = 'JFK'
OUTBOUND_DEST = 'SEA'
OUTBOUND_DATE = '2024-05-20'
INBOUND_FLIGHT_NUM = 'HAT276'
INBOUND_ORIGIN = 'SEA'
INBOUND_DEST = 'JFK'
INBOUND_DATE = '2024-05-25'
CABIN_CLASS = 'basic_economy'
EXPECTED_FREE_BAGS = 1  # retained for parity (not asserted here)

# --- Data Gathering ---
reservation_details = None
user_details = None
outbound_flights = None
inbound_flights = None
api_error = None

try:
    reservation_details = airline.get_reservation_details(reservation_id=RESERVATION_ID)
    user_details = airline.get_user_details(user_id=USER_ID)
    outbound_flights = airline.search_direct_flight(
        origin=OUTBOUND_ORIGIN, destination=OUTBOUND_DEST, date=OUTBOUND_DATE
    )
    inbound_flights = airline.search_direct_flight(
        origin=INBOUND_ORIGIN, destination=INBOUND_DEST, date=INBOUND_DATE
    )
except Exception as e:
    api_error = f"An API call failed during data gathering: {e}"

# --- Assertion 1: Reservation exists and belongs to the user ---
assertion_condition_1 = (
    reservation_details is not None and
    compare_strings(reservation_details.get('user_id', ''), USER_ID)
)
assertion_message_1 = (
    f"Assertion 1 Failed: Reservation '{RESERVATION_ID}' for user '{USER_ID}' "
    f"not found or mismatched. Found user: "
    f"{reservation_details.get('user_id') if reservation_details else 'None'}. "
    f"API Error: {api_error}"
)
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: Reservation includes passenger 'Ethan' ---
passengers_on_reservation = reservation_details.get('passengers', []) if reservation_details else []
assertion_condition_2 = any(
    compare_strings(p.get('first_name', ''), PASSENGER_TO_REMOVE)
    for p in passengers_on_reservation
)
assertion_message_2 = (
    f"Assertion 2 Failed: Passenger '{PASSENGER_TO_REMOVE}' not found in reservation "
    f"'{RESERVATION_ID}'. Passengers found: {[p.get('first_name') for p in passengers_on_reservation]}."
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Removing a single passenger is not supported (read-only policy check) ---
# Do not call mutating APIs here. Verify read-only that no removal API exists.
available_funcs = set(dir(airline))
remove_like_ops = {
    'remove_reservation_passenger', 'delete_reservation_passenger',
    'remove_passenger_from_reservation', 'delete_passenger_from_reservation',
    'remove_passenger', 'delete_passenger'
}
has_direct_remove_api = any(fname in available_funcs for fname in remove_like_ops)

assertion_condition_3 = (has_direct_remove_api is False)
assertion_message_3 = (
    "Assertion 3 Failed: Found an API that appears to remove passengers directly, "
    f"which would contradict domain policy. Found: "
    f"{sorted(list(available_funcs.intersection(remove_like_ops)))}"
)
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: User profile contains expected DOB ---
assertion_condition_4 = (
    user_details is not None and
    compare_strings(user_details.get('dob', ''), USER_DOB)
)
assertion_message_4 = (
    f"Assertion 4 Failed: User '{USER_ID}' DOB should be '{USER_DOB}'. Found: "
    f"{user_details.get('dob') if user_details else 'No user details found'}. "
    f"API Error: {api_error}"
)
assert assertion_condition_4, assertion_message_4

# --- Assertion 5: Two gift cards with sufficient balances are present ---
payment_methods = user_details.get('payment_methods', {}) if user_details else {}
card1 = payment_methods.get(GIFT_CARD_1_ID, {})
card2 = payment_methods.get(GIFT_CARD_2_ID, {})
card1_valid = (
    compare_strings(card1.get('source', ''), 'gift_card') and
    card1.get('amount', 0) >= GIFT_CARD_1_MIN_BALANCE
)
card2_valid = (
    compare_strings(card2.get('source', ''), 'gift_card') and
    card2.get('amount', 0) >= GIFT_CARD_2_MIN_BALANCE
)
assertion_condition_5 = card1_valid and card2_valid
assertion_message_5 = (
    f"Assertion 5 Failed: User '{USER_ID}' gift cards are not set up correctly. "
    f"Card1 valid: {card1_valid}, Card2 valid: {card2_valid}. Details: {payment_methods}"
)
assert assertion_condition_5, assertion_message_5

# --- Assertion 6: Outbound flight HAT069 (JFK->SEA, 2024-05-20) has basic_economy seats ---
outbound_flight = next(
    (f for f in (outbound_flights or []) if compare_strings(f.get('flight_number', ''), OUTBOUND_FLIGHT_NUM)),
    None
)
assertion_condition_6 = (
    outbound_flight is not None and
    outbound_flight.get('available_seats', {}).get(CABIN_CLASS, 0) > 0
)
assertion_message_6 = (
    f"Assertion 6 Failed: Flight '{OUTBOUND_FLIGHT_NUM}' from {OUTBOUND_ORIGIN} to "
    f"{OUTBOUND_DEST} on {OUTBOUND_DATE} in {CABIN_CLASS} is not available. API Error: {api_error}"
)
assert assertion_condition_6, assertion_message_6

# --- Assertion 7: Inbound flight HAT276 (SEA->JFK, 2024-05-25) has basic_economy seats ---
inbound_flight = next(
    (f for f in (inbound_flights or []) if compare_strings(f.get('flight_number', ''), INBOUND_FLIGHT_NUM)),
    None
)
assertion_condition_7 = (
    inbound_flight is not None and
    inbound_flight.get('available_seats', {}).get(CABIN_CLASS, 0) > 0
)
assertion_message_7 = (
    f"Assertion 7 Failed: Flight '{INBOUND_FLIGHT_NUM}' from {INBOUND_ORIGIN} to "
    f"{INBOUND_DEST} on {INBOUND_DATE} in {CABIN_CLASS} is not available. API Error: {api_error}"
)
assert assertion_condition_7, assertion_message_7


# Action

**Simulated User**: I need to remove Ethan from my reservation H9ZU1C. If that's not possible, please cancel it. I also want to book a new flight.

In [69]:
import airline

airline.get_reservation_details(reservation_id='H9ZU1C')

{'reservation_id': 'H9ZU1C',
 'user_id': 'mia_kim_4397',
 'origin': 'MIA',
 'destination': 'IAH',
 'flight_type': 'one_way',
 'cabin': 'economy',
 'flights': [{'origin': 'MIA',
   'destination': 'EWR',
   'flight_number': 'HAT192',
   'date': '2024-05-24',
   'price': 174},
  {'origin': 'EWR',
   'destination': 'IAH',
   'flight_number': 'HAT166',
   'date': '2024-05-25',
   'price': 125}],
 'passengers': [{'first_name': 'James',
   'last_name': 'Moore',
   'dob': '1977-02-03'},
  {'first_name': 'Ethan', 'last_name': 'Lopez', 'dob': '1991-11-04'},
  {'first_name': 'Amelia', 'last_name': 'Sanchez', 'dob': '1968-08-24'}],
 'payment_history': [{'payment_id': 'gift_card_7359776', 'amount': 897}],
 'created_at': '2024-05-01T04:50:18',
 'total_baggages': 0,
 'nonfree_baggages': 0,
 'insurance': 'no'}

**Action Agent**: Since I can not remove any passenger from the reservation. Moreover, since the booking does not have travel insurance, was booked more than 24 hours ago, and is in economy, I am unable to cancel it, too.

How would you like to proceed?

**Simulated User**: Okay. I'd like to book a round-trip flight from New York, either JFK or EWR, to anywhere on the West Coast. I want to leave on May 20th and return on May 25th.

In [70]:
airline.list_all_airports()

{'SFO': 'San Francisco',
 'JFK': 'New York',
 'LAX': 'Los Angeles',
 'ORD': 'Chicago',
 'DFW': 'Dallas',
 'DEN': 'Denver',
 'SEA': 'Seattle',
 'ATL': 'Atlanta',
 'MIA': 'Miami',
 'BOS': 'Boston',
 'PHX': 'Phoenix',
 'IAH': 'Houston',
 'LAS': 'Las Vegas',
 'MCO': 'Orlando',
 'EWR': 'Newark',
 'CLT': 'Charlotte',
 'MSP': 'Minneapolis',
 'DTW': 'Detroit',
 'PHL': 'Philadelphia',
 'LGA': 'LaGuardia'}

In [71]:
airline.search_direct_flight(origin='JFK', destination='SFO', date='2024-05-20')

[{'flight_number': 'HAT023',
  'origin': 'JFK',
  'destination': 'SFO',
  'scheduled_departure_time_est': '14:00:00',
  'scheduled_arrival_time_est': '20:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 20, 'economy': 13, 'business': 11},
  'prices': {'basic_economy': 100, 'economy': 191, 'business': 473}}]

In [72]:
airline.search_direct_flight(origin='JFK', destination='LAX', date='2024-05-20')

[]

In [73]:
airline.search_direct_flight(origin='JFK', destination='SEA', date='2024-05-20')

[{'flight_number': 'HAT069',
  'origin': 'JFK',
  'destination': 'SEA',
  'scheduled_departure_time_est': '06:00:00',
  'scheduled_arrival_time_est': '12:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 17, 'economy': 12, 'business': 3},
  'prices': {'basic_economy': 51, 'economy': 121, 'business': 239}},
 {'flight_number': 'HAT083',
  'origin': 'JFK',
  'destination': 'SEA',
  'scheduled_departure_time_est': '01:00:00',
  'scheduled_arrival_time_est': '07:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 16, 'economy': 7, 'business': 3},
  'prices': {'basic_economy': 87, 'economy': 100, 'business': 276}}]

In [74]:
airline.search_direct_flight(origin='EWR', destination='SFO', date='2024-05-20')

[]

In [75]:
airline.search_direct_flight(origin='EWR', destination='LAX', date='2024-05-20')

[{'flight_number': 'HAT041',
  'origin': 'EWR',
  'destination': 'LAX',
  'scheduled_departure_time_est': '07:00:00',
  'scheduled_arrival_time_est': '12:30:00',
  'status': 'available',
  'available_seats': {'basic_economy': 9, 'economy': 10, 'business': 16},
  'prices': {'basic_economy': 86, 'economy': 196, 'business': 306}}]

In [76]:
airline.search_direct_flight(origin='EWR', destination='SEA', date='2024-05-20')

[]

In [77]:
airline.search_direct_flight(origin='SFO', destination='JFK', date='2024-05-25')

[]

In [78]:
airline.search_direct_flight(origin='LAX', destination='EWR', date='2024-05-25')

[{'flight_number': 'HAT012',
  'origin': 'LAX',
  'destination': 'EWR',
  'scheduled_departure_time_est': '12:00:00',
  'scheduled_arrival_time_est': '17:30:00',
  'status': 'available',
  'available_seats': {'basic_economy': 17, 'economy': 18, 'business': 10},
  'prices': {'basic_economy': 77, 'economy': 184, 'business': 253}},
 {'flight_number': 'HAT228',
  'origin': 'LAX',
  'destination': 'EWR',
  'scheduled_departure_time_est': '20:00:00',
  'scheduled_arrival_time_est': '01:30:00+1',
  'status': 'available',
  'available_seats': {'basic_economy': 19, 'economy': 16, 'business': 19},
  'prices': {'basic_economy': 85, 'economy': 105, 'business': 488}}]

In [79]:
airline.search_direct_flight(origin='SEA', destination='JFK', date='2024-05-25')

[{'flight_number': 'HAT021',
  'origin': 'SEA',
  'destination': 'JFK',
  'scheduled_departure_time_est': '19:00:00',
  'scheduled_arrival_time_est': '01:00:00+1',
  'status': 'available',
  'available_seats': {'basic_economy': 15, 'economy': 14, 'business': 19},
  'prices': {'basic_economy': 60, 'economy': 198, 'business': 398}},
 {'flight_number': 'HAT089',
  'origin': 'SEA',
  'destination': 'JFK',
  'scheduled_departure_time_est': '10:00:00',
  'scheduled_arrival_time_est': '16:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 1, 'economy': 2, 'business': 15},
  'prices': {'basic_economy': 99, 'economy': 153, 'business': 377}},
 {'flight_number': 'HAT100',
  'origin': 'SEA',
  'destination': 'JFK',
  'scheduled_departure_time_est': '13:00:00',
  'scheduled_arrival_time_est': '19:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 13, 'economy': 12, 'business': 1},
  'prices': {'basic_economy': 97, 'economy': 176, 'business': 291}},
 {'fligh

**Action Agent**: I found a few round-trip options for you.

**Option 1: New York (EWR) to Los Angeles (LAX)**
*   **Departure (May 20th):** EWR -> LAX, Flight HAT041, 07:00 - 12:30
*   **Return (May 25th):**
    *   LAX -> EWR, Flight HAT012, 12:00 - 17:30
    *   LAX -> EWR, Flight HAT228, 20:00 - 01:30 (next day)

**Option 2: New York (JFK) to Seattle (SEA)**
*   **Departure (May 20th):**
    *   JFK -> SEA, Flight HAT069, 06:00 - 12:00
    *   JFK -> SEA, Flight HAT083, 01:00 - 07:00
*   **Return (May 25th):**
    *   SEA -> JFK, Flight HAT021, 19:00 - 01:00 (next day)
    *   SEA -> JFK, Flight HAT089, 10:00 - 16:00
    *   SEA -> JFK, Flight HAT100, 13:00 - 19:00
    *   SEA -> JFK, Flight HAT276, 18:00 - 00:00 (next day)

Please let me know which flights you'd like to book. I will also need to know who will be traveling.

**Simulated User**: Can you tell me the prices for these flights? I'm looking for the cheapest direct option.

**Action Agent**: Of course. Here are the prices for the cheapest options (Basic Economy):

**Option 1: New York (EWR) to Los Angeles (LAX)**
*   Departure (HAT041): \$86
*   Return (HAT012): \$77
*   **Total Round-Trip: \$163**

**Option 2: New York (JFK) to Seattle (SEA)**
*   The cheapest departure flight is HAT069 at \$51.
*   The cheapest return flight is HAT276 at \$55.
*   **Total Round-Trip: \$106**

The cheapest direct round-trip option is from New York (JFK) to Seattle (SEA) for \$106. Would you like to book this option?

**Simulated User**: Yes, please book the flight from JFK to Seattle. My user ID is mia_kim_4397. I'll need one free bag.

In [80]:
airline.get_user_details(user_id='mia_kim_4397')

{'name': {'first_name': 'Mia', 'last_name': 'Kim'},
 'address': {'address1': '681 Cedar Street',
  'address2': 'Suite 847',
  'city': 'Washington',
  'country': 'USA',
  'state': 'DC',
  'zip': '20017'},
 'email': 'mia.kim6850@example.com',
 'dob': '1965-06-09',
 'payment_methods': {'gift_card_7359776': {'source': 'gift_card',
   'amount': 39,
   'id': 'gift_card_7359776'},
  'gift_card_7773485': {'source': 'gift_card',
   'amount': 203,
   'id': 'gift_card_7773485'}},
 'saved_passengers': [{'first_name': 'Ethan',
   'last_name': 'Lopez',
   'dob': '1991-11-04'}],
 'membership': 'silver',
 'reservations': ['C4GP7M', 'XL2C75', 'H9ZU1C']}

**Action Agent**: I can book that flight for you. Who will be the passenger for this flight? Is it for you, Mia Kim, or for the saved passenger, Ethan Lopez? If it's for someone else, please provide their full name and date of birth.

**Simulated User**: The flight is for me, Mia Kim.

**Action Agent:** Great! Before I proceed with booking, could you please confirm which payment method you’d like me to use — your gift cards or another method?

**Simulated User:** Please use my gift cards as listed in my profile.

In [81]:
airline.book_reservation(
    user_id='mia_kim_4397',
    origin='JFK',
    destination='SEA',
    flight_type='round_trip',
    cabin='basic_economy',
    flights=[
        {'flight_number': 'HAT069', 'date': '2024-05-20'},
        {'flight_number': 'HAT276', 'date': '2024-05-25'}
    ],
    passengers=[
        {'first_name': 'Mia', 'last_name': 'Kim', 'dob': '1965-06-09'}
    ],
    payment_methods=[
        {'payment_id': 'gift_card_7359776', 'amount': 39.0},
        {'payment_id': 'gift_card_7773485', 'amount': 67.0}
    ],
    total_baggages=1,
    nonfree_baggages=0,
    insurance='no'
)

{'reservation_id': 'UWDKJW',
 'user_id': 'mia_kim_4397',
 'origin': 'JFK',
 'destination': 'SEA',
 'flight_type': 'round_trip',
 'cabin': 'basic_economy',
 'flights': [{'flight_number': 'HAT069',
   'date': '2024-05-20',
   'price': 51,
   'origin': 'JFK',
   'destination': 'SEA'},
  {'flight_number': 'HAT276',
   'date': '2024-05-25',
   'price': 55,
   'origin': 'SEA',
   'destination': 'JFK'}],
 'passengers': [{'first_name': 'Mia',
   'last_name': 'Kim',
   'dob': '1965-06-09'}],
 'payment_history': [{'payment_id': 'gift_card_7359776', 'amount': 39.0},
  {'payment_id': 'gift_card_7773485', 'amount': 67.0}],
 'created_at': '2024-05-15T15:00:00',
 'total_baggages': 1,
 'nonfree_baggages': 0,
 'insurance': 'no'}

**Action Agent**: I have successfully booked your flight from JFK to Seattle for you.

Here are the details:
*   **Flights:**
    *   **Departure (May 20th):** JFK to SEA, Flight HAT069
    *   **Return (May 25th):** SEA to JFK, Flight HAT276
*   **Passenger:** Mia Kim
*   **Baggage:** 1 free bag included
*   **Total Cost:** \$106, paid with your gift cards.

**Simulated User**: Great, thank you. I have a question: why are cancellations not refunded to gift cards?

In [82]:
airline.transfer_to_human_agents(summary="The user is asking why cancellations are not refunded to gift cards.")

'Transfer successful'

**Action Agent**: I understand your question. That is a great question for a human agent. I am connecting you to a human agent who can answer your question about our refund policy.

# Final Assertion

1. The reservation with ID `H9ZU1C` is not cancelled.
2. A new round-trip reservation is created for the user `mia_kim_4397` from New York (JFK) to Seattle (SEA).
3. The new reservation is for flights `HAT069` on May 20, 2024, and `HAT276` on May 25, 2024.
4. The passenger on the new reservation is Mia Kim, with the date of birth `1965-06-09`.
5. The new reservation is booked in basic economy, includes one baggage item, and has no insurance.
6. The payment for the new reservation was processed using `gift_card_7359776` for \$39 and `gift_card_7773485` for \$67.

In [83]:
import airline
from Scripts.assertions_utils import compare_strings

# --- Constants for the test scenario ---
USER_ID = "mia_kim_4397"
EXISTING_RESERVATION_ID = "H9ZU1C"
NEW_FLIGHT_ORIGIN = "JFK"
NEW_FLIGHT_DESTINATION = "SEA"
NEW_FLIGHT_DEPARTURE_DATE = "2024-05-20"
NEW_FLIGHT_RETURN_DATE = "2024-05-25"
NEW_FLIGHT_OUTBOUND_NUMBER = "HAT069"
NEW_FLIGHT_INBOUND_NUMBER = "HAT276"
PASSENGER_FIRST_NAME = "Mia"
PASSENGER_LAST_NAME = "Kim"
PASSENGER_DOB = "1965-06-09"
CABIN_CLASS = "basic_economy"
TOTAL_BAGGAGES = 1
NONFREE_BAGGAGES = 0
INSURANCE_STATUS = "no"
PAYMENT_GIFT_CARD_1_ID = "gift_card_7359776"
PAYMENT_GIFT_CARD_1_AMOUNT = 39.0
PAYMENT_GIFT_CARD_2_ID = "gift_card_7773485"
PAYMENT_GIFT_CARD_2_AMOUNT = 67.0

# --- Assertion 1: The reservation with ID `H9ZU1C` is not cancelled. ---
api_error_1 = None
try:
    existing_reservation_details = airline.get_reservation_details(reservation_id=EXISTING_RESERVATION_ID)
except Exception as e:
    existing_reservation_details = None
    api_error_1 = f"Error fetching existing reservation: {e}"

assertion_condition_1 = (
    existing_reservation_details is not None and
    (existing_reservation_details.get("reservation_status") or
     existing_reservation_details.get("status") or
     existing_reservation_details.get("state") or
     "active").lower() not in ["cancelled", "canceled"]
)
assertion_message_1 = (
    f"Assertion 1 Failed: Expected reservation '{EXISTING_RESERVATION_ID}' to remain active. "
    f"Status found: '{(existing_reservation_details.get('reservation_status') or existing_reservation_details.get('status') or existing_reservation_details.get('state')) if existing_reservation_details else 'None'}'. "
    f"{('API Error: ' + api_error_1) if api_error_1 else ''}"
)
assert assertion_condition_1, assertion_message_1

# --- Data Gathering for Assertions 2-6 ---
new_reservation_details = None
api_error_2 = None
try:
    user_details = airline.get_user_details(user_id=USER_ID)
    user_reservations = user_details.get("reservations", []) if user_details else []
    # Find a new round-trip JFK<->SEA reservation (excluding the existing one)
    for res_id in user_reservations:
        if compare_strings(res_id, EXISTING_RESERVATION_ID):
            continue
        try:
            temp_res_details = airline.get_reservation_details(reservation_id=res_id)
        except Exception:
            temp_res_details = None
        if (
            temp_res_details and
            compare_strings(temp_res_details.get("user_id", ""), USER_ID) and
            compare_strings(temp_res_details.get("flight_type", ""), "round_trip") and
            compare_strings(temp_res_details.get("origin", ""), NEW_FLIGHT_ORIGIN) and
            compare_strings(temp_res_details.get("destination", ""), NEW_FLIGHT_DESTINATION)
        ):
            new_reservation_details = temp_res_details
            break
except Exception as e:
    api_error_2 = f"Error gathering new reservation details: {e}"
    new_reservation_details = None

# --- Assertion 2: New round-trip reservation found for JFK -> SEA ---
assertion_condition_2 = (
    new_reservation_details is not None and
    compare_strings(new_reservation_details.get("user_id", ""), USER_ID) and
    compare_strings(new_reservation_details.get("flight_type", ""), "round_trip") and
    compare_strings(new_reservation_details.get("origin", ""), NEW_FLIGHT_ORIGIN) and
    compare_strings(new_reservation_details.get("destination", ""), NEW_FLIGHT_DESTINATION)
)
assertion_message_2 = (
    "Assertion 2 Failed: A new round-trip reservation was not found or its details are incorrect. "
    f"API Error: {api_error_2 if api_error_2 else 'None'}"
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Flights are exactly HAT069 (2024-05-20) and HAT276 (2024-05-25) ---
flights = new_reservation_details.get("flights", []) if new_reservation_details else []
expected_flights = {
    (NEW_FLIGHT_OUTBOUND_NUMBER, NEW_FLIGHT_DEPARTURE_DATE),
    (NEW_FLIGHT_INBOUND_NUMBER, NEW_FLIGHT_RETURN_DATE),
}
actual_flights = set((f.get("flight_number"), f.get("date")) for f in flights)

assertion_condition_3 = (actual_flights == expected_flights)
assertion_message_3 = (
    f"Assertion 3 Failed: New reservation flights mismatch. "
    f"Expected: {expected_flights}, Actual: {actual_flights}."
)
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: Passenger is exactly Mia Kim with DOB 1965-06-09 ---
passengers = new_reservation_details.get("passengers", []) if new_reservation_details else []
passenger_details_correct = False
if len(passengers) == 1:
    p = passengers[0]
    passenger_details_correct = (
        compare_strings(p.get("first_name", ""), PASSENGER_FIRST_NAME) and
        compare_strings(p.get("last_name", ""), PASSENGER_LAST_NAME) and
        compare_strings(p.get("dob", ""), PASSENGER_DOB)
    )

assertion_condition_4 = passenger_details_correct
assertion_message_4 = (
    f"Assertion 4 Failed: Passenger details incorrect. "
    f"Expected single passenger 'Mia Kim' DOB '{PASSENGER_DOB}'. "
    f"Actual: {passengers}."
)
assert assertion_condition_4, assertion_message_4

# --- Assertion 5: Cabin/basic_economy, baggage=1 (all free), insurance='no' ---
assertion_condition_5 = (
    compare_strings(new_reservation_details.get("cabin", ""), CABIN_CLASS) and
    new_reservation_details.get("total_baggages") == TOTAL_BAGGAGES and
    new_reservation_details.get("nonfree_baggages") == NONFREE_BAGGAGES and
    compare_strings(new_reservation_details.get("insurance", ""), INSURANCE_STATUS)
)
assertion_message_5 = (
    "Assertion 5 Failed: Booking details mismatch. "
    f"Cabin: {new_reservation_details.get('cabin') if new_reservation_details else None}, "
    f"Total Baggages: {new_reservation_details.get('total_baggages') if new_reservation_details else None}, "
    f"Nonfree Baggages: {new_reservation_details.get('nonfree_baggages') if new_reservation_details else None}, "
    f"Insurance: {new_reservation_details.get('insurance') if new_reservation_details else None}."
)
assert assertion_condition_5, assertion_message_5

# --- Assertion 6: Payment split exactly {gift_card_7359776: 39.0, gift_card_7773485: 67.0} ---
payment_history = new_reservation_details.get("payment_history", []) if new_reservation_details else []
expected_payments = {
    (PAYMENT_GIFT_CARD_1_ID, PAYMENT_GIFT_CARD_1_AMOUNT),
    (PAYMENT_GIFT_CARD_2_ID, PAYMENT_GIFT_CARD_2_AMOUNT),
}
actual_payments = set((p.get("payment_id"), p.get("amount")) for p in payment_history)

assertion_condition_6 = (actual_payments == expected_payments)
assertion_message_6 = (
    f"Assertion 6 Failed: Payment history mismatch. "
    f"Expected: {expected_payments}, Actual: {actual_payments}."
)
assert assertion_condition_6, assertion_message_6
